# Notebook 08 — CycleGAN Fog Generation

**Project:** Adaptive Road Vision  
**Path:** `C:\Users\ankit\Desktop\Adaptive_Road_Vision`

This notebook loads the trained CycleGAN generators from Notebook 07 and creates synthetic fog from the prepared Day and Night source domains.

### Expected experiment
- 1,500 Day → Fog images
- 1,500 Night → Fog images
- 3,000 synthetic fog images total

Validation/test Fog images are not copied into the CycleGAN target domain; the generated images are derived from the CycleGAN source images only.


In [ ]:
from pathlib import Path
import json
import random
import time

PROJECT_ROOT = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")

DAY_DIR = PROJECT_ROOT / "datasets" / "cyclegan" / "day"
NIGHT_DIR = PROJECT_ROOT / "datasets" / "cyclegan" / "night"
DAY_FOG_DIR = PROJECT_ROOT / "datasets" / "cyclegan" / "generated_day_fog"
NIGHT_FOG_DIR = PROJECT_ROOT / "datasets" / "cyclegan" / "generated_night_fog"

MODEL_DIR = PROJECT_ROOT / "models" / "cyclegan"
DAY_CKPT = MODEL_DIR / "day_to_fog" / "checkpoint_final.pth"
NIGHT_CKPT = MODEL_DIR / "night_to_fog" / "checkpoint_final.pth"

META_DIR = PROJECT_ROOT / "datasets" / "processed" / "metadata"
OUTPUT_CSV = META_DIR / "cyclegan_generated_metadata.csv"

IMG_SIZE = 256
MAX_SOURCES_PER_DOMAIN = 1500
SEED = 42

for p in [DAY_FOG_DIR, NIGHT_FOG_DIR, META_DIR]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(SEED)

print("Project root:", PROJECT_ROOT)
print("Day checkpoint:", DAY_CKPT)
print("Night checkpoint:", NIGHT_CKPT)


In [ ]:
# Verify Notebook 07 outputs and source folders

required = {
    "Day source": DAY_DIR,
    "Night source": NIGHT_DIR,
    "Day→Fog checkpoint": DAY_CKPT,
    "Night→Fog checkpoint": NIGHT_CKPT,
}

for name, path in required.items():
    print(f"{name}: {'OK' if path.exists() else 'MISSING'} -> {path}")

missing = [str(p) for p in required.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Required inputs are missing. Run Notebook 07 first and verify the "
        "CycleGAN folders/checkpoints."
    )


In [ ]:
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from torchvision import transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Using device:", DEVICE)


## 1. Recreate the Notebook 07 generator

The generator architecture below matches the ResNet-style generator used by the provided Notebook 07 workflow.


In [ ]:
class ResnetBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3, padding=0, bias=True),
            nn.InstanceNorm2d(dim),
            nn.ReLU(True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3, padding=0, bias=True),
            nn.InstanceNorm2d(dim),
        )

    def forward(self, x):
        return x + self.block(x)


class ResnetGenerator(nn.Module):
    def __init__(self, input_nc=3, output_nc=3, ngf=64, n_blocks=6):
        super().__init__()

        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(input_nc, ngf, 7, padding=0),
            nn.InstanceNorm2d(ngf),
            nn.ReLU(True),
        ]

        mult = 1
        for _ in range(2):
            layers += [
                nn.Conv2d(ngf * mult, ngf * mult * 2, 3, stride=2, padding=1),
                nn.InstanceNorm2d(ngf * mult * 2),
                nn.ReLU(True),
            ]
            mult *= 2

        for _ in range(n_blocks):
            layers.append(ResnetBlock(ngf * mult))

        for _ in range(2):
            layers += [
                nn.ConvTranspose2d(
                    ngf * mult,
                    ngf * mult // 2,
                    3,
                    stride=2,
                    padding=1,
                    output_padding=1,
                ),
                nn.InstanceNorm2d(ngf * mult // 2),
                nn.ReLU(True),
            ]
            mult //= 2

        layers += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(ngf, output_nc, 7, padding=0),
            nn.Tanh(),
        ]

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


In [ ]:
def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in [
            "G_A",
            "generator",
            "generator_state_dict",
            "state_dict",
            "netG",
        ]:
            if isinstance(checkpoint.get(key), dict):
                return checkpoint[key]

        if checkpoint and all(isinstance(v, torch.Tensor) for v in checkpoint.values()):
            return checkpoint

    raise ValueError(
        "Could not find a generator state dictionary in this checkpoint. "
        "Inspect checkpoint.keys() before continuing."
    )


def load_generator(checkpoint_path):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False
    )

    generator = ResnetGenerator().to(DEVICE)
    state_dict = extract_state_dict(checkpoint)

    cleaned = {
        (k[7:] if k.startswith("module.") else k): v
        for k, v in state_dict.items()
    }

    missing, unexpected = generator.load_state_dict(cleaned, strict=False)

    if missing:
        print("Missing keys:", missing)
    if unexpected:
        print("Unexpected keys:", unexpected)

    generator.eval()
    return generator, checkpoint


G_DAY_TO_FOG, DAY_CHECKPOINT_DATA = load_generator(DAY_CKPT)
G_NIGHT_TO_FOG, NIGHT_CHECKPOINT_DATA = load_generator(NIGHT_CKPT)

print("Day→Fog generator loaded.")
print("Night→Fog generator loaded.")


In [ ]:
# Optional checkpoint inspection

def show_checkpoint_info(name, checkpoint):
    print(f"\n{name}")
    if isinstance(checkpoint, dict):
        print("Keys:", list(checkpoint.keys())[:30])
        for k in ["epoch", "loss_G", "loss_D"]:
            if k in checkpoint:
                print(f"{k}: {checkpoint[k]}")
    else:
        print("Raw state dictionary.")

show_checkpoint_info("Day→Fog", DAY_CHECKPOINT_DATA)
show_checkpoint_info("Night→Fog", NIGHT_CHECKPOINT_DATA)


In [ ]:
to_tensor = transforms.Compose([
    transforms.Resize(
        (IMG_SIZE, IMG_SIZE),
        interpolation=transforms.InterpolationMode.BICUBIC
    ),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])


def tensor_to_pil(tensor):
    tensor = tensor.detach().cpu().clamp(-1, 1)
    tensor = (tensor + 1.0) / 2.0
    array = tensor.squeeze(0).permute(1, 2, 0).numpy()
    array = (array * 255).round().astype(np.uint8)
    return Image.fromarray(array)


def generate_one(generator, source_path, output_path):
    image = Image.open(source_path).convert("RGB")
    original_size = image.size

    x = to_tensor(image).unsqueeze(0).to(DEVICE)

    with torch.inference_mode():
        y = generator(x)

    result = tensor_to_pil(y)
    result.save(output_path, quality=95)

    return original_size, result.size


## 2. Select the source images

Notebook 07 was configured for 1,500 Day and 1,500 Night source images. If the folders contain more images, this notebook uses the first 1,500 after sorted ordering so the run remains reproducible.


In [ ]:
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def image_files(folder):
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
    )

day_sources_all = image_files(DAY_DIR)
night_sources_all = image_files(NIGHT_DIR)

print("Available Day:", len(day_sources_all))
print("Available Night:", len(night_sources_all))

if len(day_sources_all) < MAX_SOURCES_PER_DOMAIN:
    raise ValueError(
        f"Expected at least {MAX_SOURCES_PER_DOMAIN} Day images, "
        f"found {len(day_sources_all)}"
    )

if len(night_sources_all) < MAX_SOURCES_PER_DOMAIN:
    raise ValueError(
        f"Expected at least {MAX_SOURCES_PER_DOMAIN} Night images, "
        f"found {len(night_sources_all)}"
    )

day_sources = day_sources_all[:MAX_SOURCES_PER_DOMAIN]
night_sources = night_sources_all[:MAX_SOURCES_PER_DOMAIN]

print("Selected Day:", len(day_sources))
print("Selected Night:", len(night_sources))


## 3. Generate Day → Fog


In [ ]:
def generated_name(domain, source_path):
    return f"{domain}fog_{source_path.stem}.jpg"


def run_generation(generator, sources, output_dir, domain_label, checkpoint_path):
    records = []
    start = time.time()

    output_dir.mkdir(parents=True, exist_ok=True)

    for i, source_path in enumerate(sources, start=1):
        output_path = output_dir / generated_name(domain_label, source_path)

        original_size, generated_size = generate_one(
            generator,
            source_path,
            output_path
        )

        records.append({
            "generated_image": output_path.name,
            "generated_path": str(output_path),
            "source_image": source_path.name,
            "source_path": str(source_path),
            "source_id": source_path.stem,
            "source_domain": domain_label,
            "target_condition": "Fog",
            "generation_model": f"{domain_label}_to_fog",
            "checkpoint": str(checkpoint_path),
            "original_width": original_size[0],
            "original_height": original_size[1],
            "generated_width": generated_size[0],
            "generated_height": generated_size[1],
        })

        if i == 1 or i % 100 == 0 or i == len(sources):
            elapsed = time.time() - start
            print(f"{domain_label}: {i}/{len(sources)} | {elapsed:.1f}s")

    return records


day_records = run_generation(
    G_DAY_TO_FOG,
    day_sources,
    DAY_FOG_DIR,
    "day",
    DAY_CKPT,
)

print("Day→Fog complete:", len(day_records))


## 4. Generate Night → Fog


In [ ]:
night_records = run_generation(
    G_NIGHT_TO_FOG,
    night_sources,
    NIGHT_FOG_DIR,
    "night",
    NIGHT_CKPT,
)

print("Night→Fog complete:", len(night_records))


In [ ]:
import pandas as pd

generated_df = pd.DataFrame(day_records + night_records)
generated_df["generation_index"] = range(1, len(generated_df) + 1)

generated_df.to_csv(OUTPUT_CSV, index=False)

print("Saved metadata:", OUTPUT_CSV)
print("Total generated:", len(generated_df))
print(generated_df["source_domain"].value_counts())


## 5. Verify the generated dataset

Expected:
- 1,500 Day → Fog
- 1,500 Night → Fog
- 3,000 total


In [ ]:
day_outputs = image_files(DAY_FOG_DIR)
night_outputs = image_files(NIGHT_FOG_DIR)

print("Day→Fog files:", len(day_outputs))
print("Night→Fog files:", len(night_outputs))
print("Total:", len(day_outputs) + len(night_outputs))

assert len(day_outputs) == 1500
assert len(night_outputs) == 1500
assert len(generated_df) == 3000

missing = [
    p for p in generated_df["generated_path"]
    if not Path(p).exists()
]

assert not missing, f"Missing generated files: {len(missing)}"
assert generated_df["generated_image"].is_unique
assert generated_df["source_id"].notna().all()

print("All generation checks: PASS")


## 6. Preview source vs generated fog

This is a quick sanity check only. Detailed quality filtering belongs in Notebook 09.


In [ ]:
import matplotlib.pyplot as plt

def preview_pairs(source_paths, output_dir, domain, n=5):
    selected = list(source_paths)[:n]

    fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))
    if n == 1:
        axes = np.array([axes])

    for row, source_path in enumerate(selected):
        output_path = output_dir / generated_name(domain, source_path)

        source_img = Image.open(source_path).convert("RGB")
        generated_img = Image.open(output_path).convert("RGB")

        axes[row, 0].imshow(source_img)
        axes[row, 0].set_title(f"Original {domain}: {source_path.name}")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(generated_img)
        axes[row, 1].set_title(f"Generated Fog: {output_path.name}")
        axes[row, 1].axis("off")

    plt.tight_layout()
    plt.show()


preview_pairs(day_sources, DAY_FOG_DIR, "day")
preview_pairs(night_sources, NIGHT_FOG_DIR, "night")


In [ ]:
# Save an experiment summary

summary = {
    "project_root": str(PROJECT_ROOT),
    "source_day_count": len(day_sources),
    "source_night_count": len(night_sources),
    "generated_day_fog_count": len(day_outputs),
    "generated_night_fog_count": len(night_outputs),
    "total_generated_fog_count": len(day_outputs) + len(night_outputs),
    "image_size": IMG_SIZE,
    "device": str(DEVICE),
    "day_checkpoint": str(DAY_CKPT),
    "night_checkpoint": str(NIGHT_CKPT),
    "metadata_csv": str(OUTPUT_CSV),
    "seed": SEED,
    "note": (
        "Synthetic fog is generated from the CycleGAN source domains. "
        "Validation/test Fog images are not used as CycleGAN training targets."
    ),
}

summary_path = PROJECT_ROOT / "results" / "cyclegan" / "fog_generation_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("Saved:", summary_path)


# Notebook 08 complete

### Outputs

```text
datasets/
└── cyclegan/
    ├── generated_day_fog/       # 1,500 images
    └── generated_night_fog/     # 1,500 images

datasets/processed/metadata/
└── cyclegan_generated_metadata.csv

results/cyclegan/
└── fog_generation_summary.json
```

**Next:** Notebook 09 — `09_synthetic_fog_quality_check.ipynb`
